In [ ]:
################################################################################
##  CLIP GDF
################################################################################
from typing import Union
from shapely.geometry.base import BaseGeometry
import geopandas as gpd


def clip_gdf_to_mask_func(
    gdf: gpd.GeoDataFrame,
    mask: Union[gpd.GeoDataFrame, gpd.GeoSeries, BaseGeometry],
    *,
    keep_geom_type: bool = True,
    drop_empty: bool = True,
) -> gpd.GeoDataFrame:
    """
    Keep only features from `gdf` that fall within the border of `mask`,
    and clip their geometries to the mask boundary.

    - Requires matching CRS between `gdf` and `mask` when `mask` is a GeoDataFrame/GeoSeries.
    - Returns a GeoDataFrame in the same CRS as `gdf`.

    Parameters
    ----------
    gdf : GeoDataFrame
        Input polygons (or other geometries).
    mask : GeoDataFrame | GeoSeries | shapely geometry
        Clipping mask geometry(ies). If GeoDataFrame/GeoSeries, all geometries are
        unary-unioned into a single mask geometry.
    keep_geom_type : bool
        Passed to geopandas.clip (keeps only original geometry types).
    drop_empty : bool
        If True, remove empty geometries after clipping.

    Returns
    -------
    GeoDataFrame
        Clipped subset of `gdf`.
    """
    if gdf.crs is None:
        raise ValueError("Input gdf has no CRS set (gdf.crs is None).")

    # Build a single mask geometry, and validate CRS if applicable
    if isinstance(mask, gpd.GeoDataFrame):
        if mask.crs is None:
            raise ValueError("Mask GeoDataFrame has no CRS set (mask.crs is None).")
        if mask.crs != gdf.crs:
            raise ValueError(f"CRS mismatch: gdf.crs={gdf.crs!r} vs mask.crs={mask.crs!r}")
        mask_geom = mask.geometry.unary_union

    elif isinstance(mask, gpd.GeoSeries):
        if mask.crs is None:
            raise ValueError("Mask GeoSeries has no CRS set (mask.crs is None).")
        if mask.crs != gdf.crs:
            raise ValueError(f"CRS mismatch: gdf.crs={gdf.crs!r} vs mask.crs={mask.crs!r}")
        mask_geom = mask.unary_union

    else:
        # Shapely geometry; assume it's already in the same CRS as gdf
        mask_geom = mask

    if mask_geom is None:
        raise ValueError("Mask geometry is None.")
    if mask_geom.is_empty:
        raise ValueError("Mask geometry is empty.")

    clipped = gpd.clip(gdf, mask_geom, keep_geom_type=keep_geom_type)

    if drop_empty:
        clipped = clipped.loc[~clipped.geometry.is_empty].copy()

    return clipped

## Importing functions

In [1]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().resolve()))
from functions._00_common_funcs import (
    batch_func,
    check_directories_func,
    path_func,
    check_gpkg_layers_func,
    import_gpkg_func,
    import_xlsx_func,
    print_raster_info,
    import_raster_func,
    quick_rstr_plot_func,
    check_processing,
    show_nameconvention_func,
)

from functions._06_training_validation_split_funcs import (
    cut_gdf_to_raster_pixels,
    build_gpkg_from_idx_df_func,
    clip_gdf_to_mask_func,
    tmp_1_idx_df_func,
    WD_idx_df_func,
    HT_idx_df_func,
    pixel_count_df_func,
    pixel_vis_func,
    training_validation_idx_split_func,
    from_idx_to_carto_gdf_func,
    picky_carto_func
)    

## Setting directory

In [2]:
notebook_dir = Path.cwd()

check_directories_func(notebook_dir)

'Functions, Dependencies, and Output directories found'

In [3]:
# years definition
tmp_1_year = ['2010-2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
years_2018_2024 = [2018, 2019, 2020, 2021, 2022, 2023, 2024]

In [4]:
from functions._00_file_paths import (
    # --- sentinel 2 basisraster ---
    s2_basisraster__10m__RD__rstr_path,
    s2_basisraster__20m__RD__rstr_path,

    
    # --- Habitat kart plusOW GPKGs ---
    # merged
    habitat_kart_plusOW__merged__gelderland__RD__gpkg_path,
    habitat_kart_plusOW__merged__website_plusHR__RD__gpkg_path,
    # pixeled
    habitat_kart_plusOW__pixeled__gelderland__RD__gpkg_path,
    habitat_kart_plusOW__pixeled__website_plusHR__RD__gpkg_path,


    # --- idx plusOW dfs ---
    # WD plusOW
    idx__WD_plusOW__division__gelderland__tmp1__df_path,
    idx__WD_plusOW__division__website_plusHR__tmp1__df_path,
    # s1 plusOW
    idx__s1_plusOW__division__gelderland__tmp1__df_path,
    idx__s1_plusOW__division__website_plusHR__tmp1__df_path,


    # --- Habitat kart fully Pixeled GPKGs
    habitat_kart_plusOW__pixeled__gelderland__RD__gpkg_path,
    habitat_kart_plusOW__pixeled__website_plusHR__RD__gpkg_path,
    

    # --- Veluwe Polygons ---
    veluwe_polygon__n2000__RD__gpkg_path,
    veluwe_polygon__ruim__RD__gpkg_path,
    veluwe_polygon__aoi__RD__gpkg_path,


    # --- Selection reference dfs ---
    habitat_reference__HB__df_path,
    habitat_reference__s1__df_path,
)

## Loading packages

In [5]:
import pandas as pd

## Importing files

In [6]:
# --- Habitat kart Processed GPKGs ---
habitat_kart_plusOW_cols_to_keep = ['years', 'habitatType1', 'habitatnaam_1_disp', 'bedekkingsOppervlakte1', 'geometry']

# Gelderland 
habitat_kart_plusOW__merged__gelderland__RD__gdf = import_gpkg_func(
    gdf_path=habitat_kart_plusOW__merged__gelderland__RD__gpkg_path,
    layer='habitat_kart_plusOW__merged__gelderland__RD__layer',
    index_col='index',
    cols_to_keep=habitat_kart_plusOW_cols_to_keep,
)

# Website plus hr 
habitat_kart_plusOW__merged__website_plusHR__RD__gdf = import_gpkg_func(
    gdf_path=habitat_kart_plusOW__merged__website_plusHR__RD__gpkg_path,
    layer='habitat_kart_plusOW__merged__website_plusHR__RD__layer',
    index_col='index',
    cols_to_keep=habitat_kart_plusOW_cols_to_keep,
)

In [7]:
# --- IDX plusOW dfs ---
# WD plusOW tmp1 IDX
idx__WD_plusOW__division__gelderland__tmp1__df = pd.read_pickle(idx__WD_plusOW__division__gelderland__tmp1__df_path)
idx__WD_plusOW__division__website_plusHR__tmp1__df = pd.read_pickle(idx__WD_plusOW__division__website_plusHR__tmp1__df_path)

# s1 plusOW tmp1 IDX
idx__s1_plusOW__division__gelderland__tmp1__df = pd.read_pickle(idx__s1_plusOW__division__gelderland__tmp1__df_path)
idx__s1_plusOW__division__website_plusHR__tmp1__df = pd.read_pickle(idx__s1_plusOW__division__website_plusHR__tmp1__df_path)

In [8]:
# --- Veluwe Polygons ---
veluwe_polygon__n2000__RD__gdf = import_gpkg_func(
    gdf_path=veluwe_polygon__n2000__RD__gpkg_path,
    layer='natura2000',
)

veluwe_polygon__ruim__RD__gdf = import_gpkg_func(
    gdf_path=veluwe_polygon__ruim__RD__gpkg_path,
    layer='ruime_veluwe_cut_RD',
)

veluwe_polygon__aoi__RD__gdf = import_gpkg_func(
    gdf_path=veluwe_polygon__aoi__RD__gpkg_path,
    layer='vennen_interesse_RD',
    cols_to_keep=['Naam', 'geometry'],
)

In [9]:
# --- Selection reference dfs ---
habitat_reference__HB__df = pd.read_pickle(habitat_reference__HB__df_path)
habitat_reference__s1__df = pd.read_pickle(habitat_reference__s1__df_path)

In [10]:
habitat_kart_plusOW__merged__gelderland__RD__gdf

,years,habitatType1,habitatnaam_1_disp,bedekkingsOppervlakte1,geometry
index,,,,,
GI_1,2013,H4030,Droge heiden,1392.185413,"MULTIPOLYGON (((179904.072 453848.68, 179909.2..."
GI_2,2013,H0000,Geen habitattype,3130.353140,"MULTIPOLYGON (((182459.684 449821.826, 182465...."
GI_3,2013,H2330,Zandverstuivingen,224.197501,"MULTIPOLYGON (((181966.179 451566.285, 181962...."
GI_4,2013,H0000,Geen habitattype,625.209744,"MULTIPOLYGON (((181831.277 454198.571, 181836...."
GI_5,2013,H0000,Geen habitattype,735.745230,"MULTIPOLYGON (((182450.03 452957.745, 182453.4..."
...,...,...,...,...,...
lgn_plusOW_2024_31607,2024,LGN_OW,Open water,0.000627,"MULTIPOLYGON (((183620 442380, 183619.974 4423..."
lgn_plusOW_2024_31608,2024,LGN_OW,Open water,19.695964,"MULTIPOLYGON (((183625 442388.464, 183624.224 ..."
lgn_plusOW_2024_31609,2024,LGN_OW,Open water,1.722867,"MULTIPOLYGON (((183625 442385, 183623.439 4423..."


## Create Pixel polygons from the raster grid

_From discussions with representatives of Provincie Gelderland, the assumption was made that the following two datasets represent the truth:_ <br>
- Gelderland <br>
- Website minHR

In [11]:
# habitat_kart_plusOW__pixeled__gelderland__RD__gdf = cut_gdf_to_raster_pixels(
#     gdf=habitat_kart_plusOW__merged__gelderland__RD__gdf,
#     raster_path=s2_basisraster__10m__RD__rstr_path,
#     all_touched=False,
#     make_geoms_valid=False,
#     progress=True,
# )

In [12]:
# habitat_kart_plusOW__pixeled__gelderland__RD__gdf.to_file(
#     filename=habitat_kart_plusOW__pixeled__gelderland__RD__gpkg_path,
#     layer="habitat_kart_plusOW__pixeled__gelderland__RD__layer",
#     driver="GPKG"
# )

In [ ]:
habitat_kart_plusOW__pixeled__website_plusHR__RD__gdf = cut_gdf_to_raster_pixels(
    gdf=habitat_kart_plusOW__merged__website_plusHR__RD__gdf,
    raster_path=s2_basisraster__10m__RD__rstr_path,
    all_touched=False,
    make_geoms_valid=False,
    progress=True,
)

[200/186887] processed; output rows so far: 3693
[400/186887] processed; output rows so far: 13556
[600/186887] processed; output rows so far: 38273
[800/186887] processed; output rows so far: 39018
[1000/186887] processed; output rows so far: 46112
[1200/186887] processed; output rows so far: 62116
[1400/186887] processed; output rows so far: 64687
[1600/186887] processed; output rows so far: 69737
[1800/186887] processed; output rows so far: 73022
[2000/186887] processed; output rows so far: 73737
[2200/186887] processed; output rows so far: 79707
[2400/186887] processed; output rows so far: 85382
[2600/186887] processed; output rows so far: 90207
[2800/186887] processed; output rows so far: 97129
[3000/186887] processed; output rows so far: 104818
[3200/186887] processed; output rows so far: 117759
[3400/186887] processed; output rows so far: 127537
[3600/186887] processed; output rows so far: 134761
[3800/186887] processed; output rows so far: 139464
[4000/186887] processed; output

In [ ]:
habitat_kart_plusOW__pixeled__website_plusHR__RD__gdf.to_file(
    filename=habitat_kart_plusOW__pixeled__website_plusHR__RD__gpkg_path,
    layer="habitat_kart_plusOW__pixeled__website_plusHR__RD__layer",
    driver="GPKG"
)

## Due to how computationally expensive the gdf building is, the files are re-imported here

In [ ]:
# Gelderland
habitat_kart_plusOW__pixeled__gelderland__RD__gdf = import_gpkg_func(
    gdf_path=habitat_kart_plusOW__pixeled__gelderland__RD__gpkg_path,
    layer="habitat_kart_plusOW__pixeled__gelderland__RD__layer",
    index_col="new_index"
)

# Website plusHR
habitat_kart_plusOW__pixeled__website_plusHR__RD__gdf = import_gpkg_func(
    gdf_path=habitat_kart_plusOW__pixeled__website_plusHR__RD__gpkg_path,
    layer="habitat_kart_plusOW__pixeled__website_plusHR__RD__layer",
    index_col="new_index"
)

In [ ]:
## rounding the areacover column off (for the pixel count later)
# Gelderland
habitat_kart_plusOW__pixeled__gelderland__RD__gdf["bedekkingsOppervlakte1"] = (
    habitat_kart_plusOW__pixeled__gelderland__RD__gdf["bedekkingsOppervlakte1"].round().astype("Int64")
)

# Website plusHR
habitat_kart_plusOW__pixeled__website_plusHR__RD__gdf["bedekkingsOppervlakte1"] = (
    habitat_kart_plusOW__pixeled__website_plusHR__RD__gdf["bedekkingsOppervlakte1"].round().astype("Int64")
)

In [ ]:
habitat_kart_plusOW__pixeled__gelderland__RD__gdf.head()

,years,habitatType1,habitatnaam_1_disp,bedekkingsOppervlakte1,geometry
index,,,,,
GI_1_1,2013,H4030,Droge heiden,100,"MULTIPOLYGON (((179899.666 453834.591, 179899...."
GI_1_2,2013,H4030,Droge heiden,100,"MULTIPOLYGON (((179909.664 453834.591, 179909...."
GI_1_3,2013,H4030,Droge heiden,100,"MULTIPOLYGON (((179919.663 453834.591, 179919...."
GI_1_4,2013,H4030,Droge heiden,100,"MULTIPOLYGON (((179899.666 453824.592, 179899...."
GI_1_5,2013,H4030,Droge heiden,100,"MULTIPOLYGON (((179909.664 453824.592, 179909...."
...,...,...,...,...,...
LGN_OW_min_gel_2024_31606_5,2024,LGN_OW,Open water,100,"MULTIPOLYGON (((183609.139 442366.218, 183609...."
LGN_OW_min_gel_2024_31606_6,2024,LGN_OW,Open water,100,"MULTIPOLYGON (((183589.142 442356.219, 183589...."
LGN_OW_min_gel_2024_31606_7,2024,LGN_OW,Open water,100,"MULTIPOLYGON (((183599.141 442356.219, 183599...."


## Build pixeled tmp 1 idx dfs

In [ ]:
# WD tmp1
idx__WD_plusOW__division__gelderland__tmp1__df, idx__WD_plusOW__division__gelderland__tmp1__count = WD_idx_df_func(
    gdf=habitat_kart_plusOW__merged__gelderland__RD__gdf,
    idx_df=idx__AO_plusOW__division__gelderland__tmp1__df,
    habitat_types_df=habitat_reference__HB__df,
    habitat_col='habitatType1',
    wd_col='WD_division',
)

In [23]:
# --- WD+OW tmp1 idx dfs ---
AO_plus_OW_tmp_1_pixeled_gelderland_idx_df, AO_plus_OW_tmp_1_pixeled_gelderland_count = tmp_1_idx_df_func(habitat_kart_pixeled_gelderland_RD_OW_gdf)
AO_plus_OW_tmp_1_pixeled_gelderland_idx_min_hr_df, AO_plus_OW_tmp_1_pixeled_gelderland_min_hr_count = tmp_1_idx_df_func(habitat_kart_pixeled_gelderland_RD_min_hr_OW_gdf)
AO_plus_OW_tmp_1_pixeled_website_idx_df, AO_plus_OW_tmp_1_pixeled_website_count = tmp_1_idx_df_func(habitat_kart_pixeled_website_RD_OW_gdf)
AO_plus_OW_tmp_1_pixeled_website_idx_plus_hr_df, AO_plus_OW_tmp_1_pixeled_website_plus_hr_count = tmp_1_idx_df_func(habitat_kart_pixeled_website_RD_plus_hr_OW_gdf)

In [24]:
# saving the idx dfs
idx_dfs_out_dir = path_func(data_dir, "03_training_validation_data/training_validation_idx_df")

AO_plus_OW_tmp_1_pixeled_gelderland_idx_df.to_pickle(idx_dfs_out_dir / "AO_plus_OW_tmp_1_pixeled_gelderland_idx_df.pkl")
AO_plus_OW_tmp_1_pixeled_gelderland_idx_min_hr_df.to_pickle(idx_dfs_out_dir / "AO_plus_OW_tmp_1_pixeled_gelderland_idx_min_hr_df.pkl")
AO_plus_OW_tmp_1_pixeled_website_idx_df.to_pickle(idx_dfs_out_dir / "AO_plus_OW_tmp_1_pixeled_website_idx_df.pkl")
AO_plus_OW_tmp_1_pixeled_website_idx_plus_hr_df.to_pickle(idx_dfs_out_dir / "AO_plus_OW_tmp_1_pixeled_website_idx_plus_hr_df.pkl")

## Build a new WD + OW tmp 1 idx df

In [25]:
# --- WD+OW tmp1 idx dfs update ---
WD_plus_OW_tmp_1_pixeled_gelderland_idx_df, WD_plus_OW_tmp_1_pixeled_gelderland_idx_count = WD_idx_df_func(
    gdf=habitat_kart_pixeled_gelderland_RD_OW_gdf,
    idx_df=AO_plus_OW_tmp_1_pixeled_gelderland_idx_df,
    habitat_types_df=robust_names_df
)

WD_plus_OW_tmp_1_pixeled_gelderland_idx_min_hr_df, WD_plus_OW_tmp_1_pixeled_gelderland_idx_min_hr_count = WD_idx_df_func(
    gdf=habitat_kart_pixeled_gelderland_RD_min_hr_OW_gdf,
    idx_df=AO_plus_OW_tmp_1_pixeled_gelderland_idx_min_hr_df,
    habitat_types_df=robust_names_df
)

WD_plus_OW_tmp_1_pixeled_website_idx_df, WD_plus_OW_tmp_1_pixeled_website_idx_count = WD_idx_df_func(
    gdf=habitat_kart_pixeled_website_RD_OW_gdf,
    idx_df=AO_plus_OW_tmp_1_pixeled_website_idx_df,
    habitat_types_df=robust_names_df
)

WD_plus_OW_tmp_1_pixeled_website_idx_plus_hr_df, WD_plus_OW_tmp_1_pixeled_website_idx_plus_hr_count = WD_idx_df_func(
    gdf=habitat_kart_pixeled_website_RD_plus_hr_OW_gdf,
    idx_df=AO_plus_OW_tmp_1_pixeled_website_idx_plus_hr_df,
    habitat_types_df=robust_names_df
)

# --- S1 tmp1 idx dfs update ---
S1_tmp_1_pixeled_gelderland_idx_df, S1_tmp_1_pixeled_gelderland_idx_count = HT_idx_df_func(
    gdf=habitat_kart_pixeled_gelderland_RD_OW_gdf,
    idx_df=AO_plus_OW_tmp_1_pixeled_gelderland_idx_df,
    habitat_types_df=s1_names_df
)

S1_tmp_1_pixeled_gelderland_idx_min_hr_df, S1_tmp_1_pixeled_gelderland_idx_min_hr_count = HT_idx_df_func(
    gdf=habitat_kart_pixeled_gelderland_RD_min_hr_OW_gdf,
    idx_df=AO_plus_OW_tmp_1_pixeled_gelderland_idx_min_hr_df,
    habitat_types_df=s1_names_df
)

S1_tmp_1_pixeled_website_idx_df, S1_tmp_1_pixeled_website_idx_count = HT_idx_df_func(
    gdf=habitat_kart_pixeled_website_RD_OW_gdf,
    idx_df=AO_plus_OW_tmp_1_pixeled_website_idx_df,
    habitat_types_df=s1_names_df
)

S1_tmp_1_pixeled_website_idx_plus_hr_df, S1_tmp_1_pixeled_website_idx_plus_hr_count = HT_idx_df_func(
    gdf=habitat_kart_pixeled_website_RD_plus_hr_OW_gdf,
    idx_df=AO_plus_OW_tmp_1_pixeled_website_idx_plus_hr_df,
    habitat_types_df=s1_names_df
)

In [26]:
# --- saving the WD+OW tmp 1 idx dfs ---
WD_plus_OW_tmp_1_pixeled_gelderland_idx_df.to_pickle(idx_dfs_out_dir / "WD_plus_OW_tmp_1_pixeled_gelderland_idx_df.pkl")
WD_plus_OW_tmp_1_pixeled_gelderland_idx_min_hr_df.to_pickle(idx_dfs_out_dir / "WD_plus_OW_tmp_1_pixeled_gelderland_idx_min_hr_df.pkl")
WD_plus_OW_tmp_1_pixeled_website_idx_df.to_pickle(idx_dfs_out_dir / "WD_plus_OW_tmp_1_pixeled_website_idx_df.pkl")
WD_plus_OW_tmp_1_pixeled_website_idx_plus_hr_df.to_pickle(idx_dfs_out_dir / "WD_plus_OW_tmp_1_pixeled_website_idx_plus_hr_df.pkl")

# --- saving the S1 tmp 1 idx dfs ---
S1_tmp_1_pixeled_gelderland_idx_df.to_pickle(idx_dfs_out_dir / "S1_tmp_1_pixeled_gelderland_idx_df.pkl")
S1_tmp_1_pixeled_gelderland_idx_min_hr_df.to_pickle(idx_dfs_out_dir / "S1_tmp_1_pixeled_gelderland_idx_min_hr_df.pkl")
S1_tmp_1_pixeled_website_idx_df.to_pickle(idx_dfs_out_dir / "S1_tmp_1_pixeled_website_idx_df.pkl")
S1_tmp_1_pixeled_website_idx_plus_hr_df.to_pickle(idx_dfs_out_dir / "S1_tmp_1_pixeled_website_idx_plus_hr_df.pkl")

## See how many "full" pixels are left

In [27]:
# doing all pixel count calculations
# --- WD+OW tmp1 pixel counts ---
WD_plus_OW_tmp_1_gelderland_no_of_pixels_df, count = pixel_count_df_func(
    gdf=habitat_kart_pixeled_gelderland_RD_OW_gdf,
    idx_df=WD_plus_OW_tmp_1_pixeled_gelderland_idx_df,
    area_col="bedekkingsOppervlakte1",
)

WD_plus_OW_tmp_1_gelderland_min_hr_no_of_pixels_df, count = pixel_count_df_func(
    gdf=habitat_kart_pixeled_gelderland_RD_min_hr_OW_gdf,
    idx_df=WD_plus_OW_tmp_1_pixeled_gelderland_idx_min_hr_df,
    area_col="bedekkingsOppervlakte1",
)

WD_plus_OW_tmp_1_website_no_of_pixels_df, count = pixel_count_df_func(
    gdf=habitat_kart_pixeled_website_RD_OW_gdf,
    idx_df=WD_plus_OW_tmp_1_pixeled_website_idx_df,
    area_col="bedekkingsOppervlakte1",
)

WD_plus_OW_tmp_1_website_plus_hr_no_of_pixels_df, count = pixel_count_df_func(
    gdf=habitat_kart_pixeled_website_RD_plus_hr_OW_gdf,
    idx_df=WD_plus_OW_tmp_1_pixeled_website_idx_plus_hr_df,
    area_col="bedekkingsOppervlakte1",
)

# --- S1 tmp1 pixel counts ---
S1_tmp_1_gelderland_no_of_pixels_df, count = pixel_count_df_func(
    gdf=habitat_kart_pixeled_gelderland_RD_OW_gdf,
    idx_df=S1_tmp_1_pixeled_gelderland_idx_df,
    area_col="bedekkingsOppervlakte1",
)

S1_tmp_1_gelderland_min_hr_no_of_pixels_df, count = pixel_count_df_func(
    gdf=habitat_kart_pixeled_gelderland_RD_min_hr_OW_gdf,
    idx_df=S1_tmp_1_pixeled_gelderland_idx_min_hr_df,
    area_col="bedekkingsOppervlakte1",
)

S1_tmp_1_website_no_of_pixels_df, count = pixel_count_df_func(
    gdf=habitat_kart_pixeled_website_RD_OW_gdf,
    idx_df=S1_tmp_1_pixeled_website_idx_df,
    area_col="bedekkingsOppervlakte1",
)

S1_tmp_1_website_plus_hr_no_of_pixels_df, count = pixel_count_df_func(
    gdf=habitat_kart_pixeled_website_RD_plus_hr_OW_gdf,
    idx_df=S1_tmp_1_pixeled_website_idx_plus_hr_df,
    area_col="bedekkingsOppervlakte1",
)

In [28]:
pixel_vis_func(WD_plus_OW_tmp_1_gelderland_no_of_pixels_df, tmp_1_year, title="WD Gelderland")

_year_str,2010-2016,2017,2018,2019,2020,2021,2022,2023,2024,Total pixels
Dry Nature,999+,999+,999+,999+,999+,999+,9,166,,1438818
No-Habitat,999+,999+,999+,999+,999+,999+,24,999+,,5572459
Semi-Wet Nature,999+,999+,730,159,378,,,173,,10027
Wet Nature,999+,596,268,,185,,29,,,2177
Open water,2,,999+,999+,999+,999+,999+,999+,999+,45577


In [29]:
pixel_vis_func(WD_plus_OW_tmp_1_gelderland_min_hr_no_of_pixels_df, tmp_1_year, title="WD Gelderland min HR")

_year_str,2010-2016,2017,2018,2019,2020,2021,2022,2023,2024,Total pixels
Dry Nature,999+,999+,999+,999+,999+,9,166,,,1434347
No-Habitat,999+,999+,999+,999+,999+,24,999+,,,5560346
Semi-Wet Nature,999+,730,159,378,,,173,,,10027
Wet Nature,999+,268,,185,,29,,,,2177
Open water,2,3,999+,999+,999+,999+,999+,999+,999+,45577


In [30]:
pixel_vis_func(WD_plus_OW_tmp_1_website_no_of_pixels_df, tmp_1_year, title="WD Website")

_year_str,2010-2016,2017,2018,2019,2020,2021,2022,2023,2024,Total pixels
No-Habitat,999+,999+,999+,999+,999+,175,999+,999+,,6087718
Semi-Wet Nature,999+,999+,999+,886,53,26,173,999+,,50191
Dry Nature,999+,999+,999+,999+,999+,42,166,44,,1445974
Wet Nature,999+,268,999+,999+,8,46,,365,,8487
Open water,999+,3,999+,999+,999+,999+,999+,999+,999+,49942


In [31]:
pixel_vis_func(WD_plus_OW_tmp_1_website_plus_hr_no_of_pixels_df, tmp_1_year, title="WD Website plus HR")

_year_str,2010-2016,2017,2018,2019,2020,2021,2022,2023,2024,Total pixels
No-Habitat,999+,999+,999+,999+,999+,999+,175,999+,,6119634
Semi-Wet Nature,999+,999+,999+,999+,393,38,26,999+,,50191
Dry Nature,999+,999+,999+,999+,999+,999+,42,210,,1450445
Wet Nature,999+,596,268,999+,185,8,46,365,,8487
Open water,999+,111,999+,999+,999+,999+,999+,999+,999+,49656


In [44]:
pixel_vis_func(S1_tmp_1_gelderland_no_of_pixels_df, tmp_1_year, title="S1 Gelderland")

_year_str,2010-2016,2017,2018,2019,2020,2021,2022,2023,2024,Total pixels
Actieve hoogvenen (heideveentjes),446,,161,,,,,,,607
Overgangs- & trilven,13,545,,,,,,,,558
Vochtige heiden,999+,998,595,159,100,,,,,7732
Zure vennen,503,,6,,90,,29,,,628
Zwakgebufferde vennen,137,1,98,,90,,,,,326


In [47]:
pixel_vis_func(S1_tmp_1_gelderland_min_hr_no_of_pixels_df, tmp_1_year, title="S1 Gelderland Min HR")

_year_str,2010-2016,2017,2018,2019,2020,2021,2022,2023,2024,Total pixels
Actieve hoogvenen (heideveentjes),446,161,,,,,,,,607
Overgangs- & trilven,558,,,,,,,,,558
Vochtige heiden,999+,595,159,100,,,,,,7732
Zure vennen,503,6,,90,,29,,,,628
Zwakgebufferde vennen,138,98,,90,,,,,,326


In [45]:
pixel_vis_func(S1_tmp_1_website_no_of_pixels_df, tmp_1_year, title="S1 Website")

_year_str,2010-2016,2017,2018,2019,2020,2021,2022,2023,2024,Total pixels
Actieve hoogvenen (heideveentjes),441,161,,,,,,,,602
Overgangs- & trilven,558,,63,182,,,,,,803
Vochtige heiden,999+,595,502,100,,,,,,8075
Zure vennen,503,6,4,90,,29,,,,632
Zwakgebufferde vennen,106,98,272,90,,,,,,566


In [46]:
pixel_vis_func(S1_tmp_1_website_plus_hr_no_of_pixels_df, tmp_1_year, title="S1 Website Plus HR")

_year_str,2010-2016,2017,2018,2019,2020,2021,2022,2023,2024,Total pixels
Actieve hoogvenen (heideveentjes),441,,161,,,,,,,602
Overgangs- & trilven,13,545,,245,,,,,,803
Vochtige heiden,999+,998,595,502,100,,,,,8075
Zure vennen,503,,6,4,90,,29,,,632
Zwakgebufferde vennen,105,1,98,272,90,,,,,566


## GPKGs for QGIS visual inspection

In [ ]:
# WD Gelderland
habitat_kart_plusOW__WD_inspection__gelderland__RD__gdf = build_gpkg_from_idx_df_func(
    idx_df=idx__WD_plusOW__division__gelderland__tmp1__df,
    source_gdf=habitat_kart_plusOW__merged__gelderland__RD__gdf,
    out_gpkg=habitat_kart_plusOW__WD_inspection__gelderland__RD__gpkg_path,
    single_layer=True,
    single_layer_name="habitat_kart_plusOW__WD_inspection__gelderland__layer",
)

In [ ]:
# s1 Gelderland
habitat_kart_plusOW__s1_inspection__gelderland__RD__gdf = build_gpkg_from_idx_df_func(
    idx_df=idx__s1_plusOW__division__gelderland__tmp1__df,
    source_gdf=habitat_kart_plusOW__merged__gelderland__RD__gdf,
    out_gpkg=habitat_kart_plusOW__s1_inspection__gelderland__RD__gpkg_path,
    single_layer=True,
    single_layer_name="habitat_kart_plusOW__s1_inspection__gelderland__layer",
)

In [ ]:
# WD Website
habitat_kart_plusOW__WD_inspection__website_plusHR__RD__gdf = build_gpkg_from_idx_df_func(
    idx_df=idx__WD_plusOW__division__website_plusHR__tmp1__df,
    source_gdf=habitat_kart_plusOW__merged__website_plusHR__RD__gdf,
    out_gpkg=habitat_kart_plusOW__WD_inspection__website_plusHR__RD__gpkg_path,
    single_layer=True,
    single_layer_name="habitat_kart_plusOW__WD_inspection__website_plusHR__layer",
)

In [ ]:
# s1 Website
habitat_kart_plusOW__s1_inspection__website_plusHR__RD__gdf = build_gpkg_from_idx_df_func(
    idx_df=idx__s1_plusOW__division__website_plusHR__tmp1__df,
    source_gdf=habitat_kart_plusOW__merged__website_plusHR__RD__gdf,
    out_gpkg=habitat_kart_plusOW__s1_inspection__website_plusHR__RD__gpkg_path,
    single_layer=True,
    single_layer_name="habitat_kart_plusOW__s1_inspection__website_plusHR__layer",
)

## Split the polygons into training and validation

In [35]:
# --- WD Gelderland ---
train_val_test_WD_plus_OW_tmp_1_gelderland_idx_df = training_validation_idx_split_func(
    df=WD_plus_OW_tmp_1_pixeled_gelderland_idx_df,
    cols=["2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"],
    train_pct=0.7,
    val_pct=0.3,
    test_pct=0.0,
    seed=42
)

# --- WD Gelderland min hr ---
train_val_test_WD_plus_OW_tmp_1_gelderland_idx_min_hr_df = training_validation_idx_split_func(
    df=WD_plus_OW_tmp_1_pixeled_gelderland_idx_min_hr_df,
    cols=["2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"],
    train_pct=0.7,
    val_pct=0.3,
    test_pct=0.0,
    seed=42
)

# --- WD Website ---
train_val_test_WD_plus_OW_tmp_1_website_idx_df = training_validation_idx_split_func(
    df=WD_plus_OW_tmp_1_pixeled_website_idx_df,
    cols=["2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"],
    train_pct=0.7,
    val_pct=0.3,
    test_pct=0.0,
    seed=42
)

# --- WD Website plus hr ---
train_val_test_WD_plus_OW_tmp_1_website_plus_hr_idx_df = training_validation_idx_split_func(
    df=WD_plus_OW_tmp_1_pixeled_website_idx_plus_hr_df,
    cols=["2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"],
    train_pct=0.7,
    val_pct=0.3,
    test_pct=0.0,
    seed=42
)

In [36]:
# --- S1 Gelderland ---
train_val_test_S1_tmp_1_gelderland_idx_df = training_validation_idx_split_func(
    df=S1_tmp_1_pixeled_gelderland_idx_df,
    cols=["2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"],
    train_pct=0.7,
    val_pct=0.3,
    test_pct=0.0,
    seed=42
)

# --- S1 Gelderland min hr ---
train_val_test_S1_tmp_1_gelderland_idx_min_hr_df = training_validation_idx_split_func(
    df=S1_tmp_1_pixeled_gelderland_idx_min_hr_df,
    cols=["2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"],
    train_pct=0.7,
    val_pct=0.3,
    test_pct=0.0,
    seed=42
)

# --- S1 Website ---
train_val_test_S1_tmp_1_website_idx_df = training_validation_idx_split_func(
    df=S1_tmp_1_pixeled_website_idx_df,
    cols=["2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"],
    train_pct=0.7,
    val_pct=0.3,
    test_pct=0.0,
    seed=42
)

# --- S1 Website plus hr ---
train_val_test_S1_tmp_1_website_plus_hr_idx_df = training_validation_idx_split_func(
    df=S1_tmp_1_pixeled_website_idx_plus_hr_df,
    cols=["2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"],
    train_pct=0.7,
    val_pct=0.3,
    test_pct=0.0,
    seed=42
)

## Prepare for training in Carto

In [37]:
# Setting output dir
# --- WD + OW tmp1 gelderland ---
train_val_test_WD_plus_OW_tmp_1_gelderland_output_dir = path_func(data_dir, "03_training_validation_data/training_validation_gpkg/WD_plus_OW_tmp_1_gelderland")
train_val_test_WD_plus_OW_tmp_1_gelderland_min_hr_output_dir = path_func(data_dir, "03_training_validation_data/training_validation_gpkg/WD_plus_OW_tmp_1_gelderland_min_hr")
train_val_test_WD_plus_OW_tmp_1_website_output_dir = path_func(data_dir, "03_training_validation_data/training_validation_gpkg/WD_plus_OW_tmp_1_website")
train_val_test_WD_plus_OW_tmp_1_website_plus_hr_output_dir = path_func(data_dir, "03_training_validation_data/training_validation_gpkg/WD_plus_OW_tmp_1_website_plus_hr")

# --- S1 tmp1 gelderland ---
train_val_test_S1_tmp_1_gelderland_output_dir = path_func(data_dir, "03_training_validation_data/training_validation_gpkg/S1_tmp_1_gelderland")
train_val_test_S1_tmp_1_gelderland_min_hr_output_dir = path_func(data_dir, "03_training_validation_data/training_validation_gpkg/S1_tmp_1_gelderland_min_hr")
train_val_test_S1_tmp_1_website_output_dir = path_func(data_dir, "03_training_validation_data/training_validation_gpkg/S1_tmp_1_website")
train_val_test_S1_tmp_1_website_plus_hr_output_dir = path_func(data_dir, "03_training_validation_data/training_validation_gpkg/S1_tmp_1_website_plus_hr")

In [38]:
# Building the train/val/test gdfs
# --- WD Gelderland ---
train_val_test_gdfs_WD_gelderland = from_idx_to_carto_gdf_func(
    training_df=train_val_test_WD_plus_OW_tmp_1_gelderland_idx_df,
    habitat_gdf=habitat_kart_pixeled_gelderland_RD_OW_gdf,
    split_cols=["train", "val"],
    dissolve_polygons=True
)

train_gdf_WD_plus_OW_tmp_1_gelderland = train_val_test_gdfs_WD_gelderland["train"]
val_gdf_WD_plus_OW_tmp_1_gelderland = train_val_test_gdfs_WD_gelderland["val"]

# --- WD Gelderland min hr ---
train_val_test_gdfs_WD_gelderland_min_hr = from_idx_to_carto_gdf_func(
    training_df=train_val_test_WD_plus_OW_tmp_1_gelderland_idx_min_hr_df,
    habitat_gdf=habitat_kart_pixeled_gelderland_RD_min_hr_OW_gdf,
    split_cols=["train", "val"],
    dissolve_polygons=True
)

train_gdf_WD_plus_OW_tmp_1_gelderland_min_hr = train_val_test_gdfs_WD_gelderland_min_hr["train"]
val_gdf_WD_plus_OW_tmp_1_gelderland_min_hr = train_val_test_gdfs_WD_gelderland_min_hr["val"]

# --- WD Website ---
train_val_test_gdfs_WD_website = from_idx_to_carto_gdf_func(
    training_df=train_val_test_WD_plus_OW_tmp_1_website_idx_df,
    habitat_gdf=habitat_kart_pixeled_website_RD_OW_gdf,
    split_cols=["train", "val"],
    dissolve_polygons=True
)

train_gdf_WD_plus_OW_tmp_1_website = train_val_test_gdfs_WD_website["train"]
val_gdf_WD_plus_OW_tmp_1_website = train_val_test_gdfs_WD_website["val"]

# --- WD Website plus HR ---
train_val_test_gdfs_WD_website_plus_hr = from_idx_to_carto_gdf_func(
    training_df=train_val_test_WD_plus_OW_tmp_1_website_plus_hr_idx_df,
    habitat_gdf=habitat_kart_pixeled_website_RD_plus_hr_OW_gdf,
    split_cols=["train", "val"],
    dissolve_polygons=True
)

train_gdf_WD_plus_OW_tmp_1_website_plus_hr = train_val_test_gdfs_WD_website_plus_hr["train"]
val_gdf_WD_plus_OW_tmp_1_website_plus_hr = train_val_test_gdfs_WD_website_plus_hr["val"]

In [39]:
# Building the train/val/test gdfs
# --- S1 Gelderland ---
train_val_test_gdfs_S1_gelderland = from_idx_to_carto_gdf_func(
    training_df=train_val_test_S1_tmp_1_gelderland_idx_df,
    habitat_gdf=habitat_kart_pixeled_gelderland_RD_OW_gdf,
    split_cols=["train", "val"],
    dissolve_polygons=True
)

train_gdf_S1_tmp_1_gelderland = train_val_test_gdfs_S1_gelderland["train"]
val_gdf_S1_tmp_1_gelderland = train_val_test_gdfs_S1_gelderland["val"]

# --- S1 Gelderland min hr ---
train_val_test_gdfs_S1_gelderland_min_hr = from_idx_to_carto_gdf_func(
    training_df=train_val_test_S1_tmp_1_gelderland_idx_min_hr_df,
    habitat_gdf=habitat_kart_pixeled_gelderland_RD_min_hr_OW_gdf,
    split_cols=["train", "val"],
    dissolve_polygons=True
)

train_gdf_S1_tmp_1_gelderland_min_hr = train_val_test_gdfs_S1_gelderland_min_hr["train"]
val_gdf_S1_tmp_1_gelderland_min_hr = train_val_test_gdfs_S1_gelderland_min_hr["val"]

# --- S1 Website ---
train_val_test_gdfs_S1_website = from_idx_to_carto_gdf_func(
    training_df=train_val_test_S1_tmp_1_website_idx_df,
    habitat_gdf=habitat_kart_pixeled_website_RD_OW_gdf,
    split_cols=["train", "val"],
    dissolve_polygons=True
)

train_gdf_S1_tmp_1_website = train_val_test_gdfs_S1_website["train"]
val_gdf_S1_tmp_1_website = train_val_test_gdfs_S1_website["val"]

# --- S1 Website plus HR ---
train_val_test_gdfs_S1_website_plus_hr = from_idx_to_carto_gdf_func(
    training_df=train_val_test_S1_tmp_1_website_plus_hr_idx_df,
    habitat_gdf=habitat_kart_pixeled_website_RD_plus_hr_OW_gdf,
    split_cols=["train", "val"],
    dissolve_polygons=True
)

train_gdf_S1_tmp_1_website_plus_hr = train_val_test_gdfs_S1_website_plus_hr["train"]
val_gdf_S1_tmp_1_website_plus_hr = train_val_test_gdfs_S1_website_plus_hr["val"]

## Saving the GDFS

In [40]:
# Saving the training gdfs as separate years
# --- WD Gelderland ---
train_WD_plus_OW_tmp_1_gelderland_year_files = picky_carto_func(
    gdf=train_gdf_WD_plus_OW_tmp_1_gelderland,
    path=train_val_test_WD_plus_OW_tmp_1_gelderland_output_dir,
    filename_base="train_WD_plus_OW_tmp_1_gelderland"
)

# --- WD Gelderland min hr ---
train_WD_plus_OW_tmp_1_gelderland_min_hr_year_files = picky_carto_func(
    gdf=train_gdf_WD_plus_OW_tmp_1_gelderland_min_hr,
    path=train_val_test_WD_plus_OW_tmp_1_gelderland_min_hr_output_dir,
    filename_base="train_WD_plus_OW_tmp_1_gelderland_min_hr"
)

# --- WD Website ---
train_WD_plus_OW_tmp_1_website_year_files = picky_carto_func(
    gdf=train_gdf_WD_plus_OW_tmp_1_website,
    path=train_val_test_WD_plus_OW_tmp_1_website_output_dir,
    filename_base="train_WD_plus_OW_tmp_1_website"
)

# --- WD Website plus HR ---
train_WD_plus_OW_tmp_1_website_plus_hr_year_files = picky_carto_func(
    gdf=train_gdf_WD_plus_OW_tmp_1_website_plus_hr,
    path=train_val_test_WD_plus_OW_tmp_1_website_plus_hr_output_dir,
    filename_base="train_WD_plus_OW_tmp_1_website_plus_hr"
)

In [41]:
# Saving the training gdfs as separate years
# --- S1 Gelderland ---
train_S1_tmp_1_gelderland_year_files = picky_carto_func(
    gdf=train_gdf_S1_tmp_1_gelderland,
    path=train_val_test_S1_tmp_1_gelderland_output_dir,
    filename_base="train_S1_tmp_1_gelderland"
)

# --- S1 Gelderland min hr ---
train_S1_tmp_1_gelderland_min_hr_year_files = picky_carto_func(
    gdf=train_gdf_S1_tmp_1_gelderland_min_hr,
    path=train_val_test_S1_tmp_1_gelderland_min_hr_output_dir,
    filename_base="train_S1_tmp_1_gelderland_min_hr"
)

# --- S1 Website ---
train_S1_tmp_1_website_year_files = picky_carto_func(
    gdf=train_gdf_S1_tmp_1_website,
    path=train_val_test_S1_tmp_1_website_output_dir,
    filename_base="train_S1_tmp_1_website"
)

# --- S1 Website plus HR ---
train_S1_tmp_1_website_plus_hr_year_files = picky_carto_func(
    gdf=train_gdf_S1_tmp_1_website_plus_hr,
    path=train_val_test_S1_tmp_1_website_plus_hr_output_dir,
    filename_base="train_S1_tmp_1_website_plus_hr"
)

In [42]:
# Saving the validation gdfs
# --- WD Gelderland ---
val_gdf_WD_plus_OW_tmp_1_gelderland.to_file(
    train_val_test_WD_plus_OW_tmp_1_gelderland_output_dir / "val_WD_plus_OW_tmp_1_gelderland.gpkg",
    layer="val_WD_plus_OW_tmp_1_gelderland",
    driver="GPKG"
)

# --- WD Gelderland min hr ---
val_gdf_WD_plus_OW_tmp_1_gelderland_min_hr.to_file(
    train_val_test_WD_plus_OW_tmp_1_gelderland_min_hr_output_dir / "val_WD_plus_OW_tmp_1_gelderland_min_hr.gpkg",
    layer="val_WD_plus_OW_tmp_1_gelderland_min_hr",
    driver="GPKG"
)

# --- WD Website ---
val_gdf_WD_plus_OW_tmp_1_website.to_file(
    train_val_test_WD_plus_OW_tmp_1_website_output_dir / "val_WD_plus_OW_tmp_1_website.gpkg",
    layer="val_WD_plus_OW_tmp_1_website",
    driver="GPKG"
)

# --- WD Website plus HR ---
val_gdf_WD_plus_OW_tmp_1_website_plus_hr.to_file(
    train_val_test_WD_plus_OW_tmp_1_website_plus_hr_output_dir / "val_WD_plus_OW_tmp_1_website_plus_hr.gpkg",
    layer="val_WD_plus_OW_tmp_1_website_plus_hr",
    driver="GPKG"
)

In [43]:
# Saving the validation gdfs
# --- S1 Gelderland ---
val_gdf_S1_tmp_1_gelderland.to_file(
    train_val_test_S1_tmp_1_gelderland_output_dir / "val_S1_tmp_1_gelderland.gpkg",
    layer="val_S1_tmp_1_gelderland",
    driver="GPKG"
)

# --- S1 Gelderland min hr ---
val_gdf_S1_tmp_1_gelderland_min_hr.to_file(
    train_val_test_S1_tmp_1_gelderland_min_hr_output_dir / "val_S1_tmp_1_gelderland_min_hr.gpkg",
    layer="val_S1_tmp_1_gelderland_min_hr",
    driver="GPKG"
)

# --- S1 Website ---
val_gdf_S1_tmp_1_website.to_file(
    train_val_test_S1_tmp_1_website_output_dir / "val_S1_tmp_1_website.gpkg",
    layer="val_S1_tmp_1_website",
    driver="GPKG"
)

# --- S1 Website plus HR ---
val_gdf_S1_tmp_1_website_plus_hr.to_file(
    train_val_test_S1_tmp_1_website_plus_hr_output_dir / "val_S1_tmp_1_website_plus_hr.gpkg",
    layer="val_S1_tmp_1_website_plus_hr",
    driver="GPKG"
)

End of the Notebook